In [11]:
# %%
import xarray as xr
import cfgrib
import numpy as np
from datetime import datetime
import glob

def gfs_ensemble_grib_to_netcdf(input_file, output_file, 
                              bbox=None, variables=None, 
                              levels=None
                              ) :

    
    # Open the GRIB file using cfgrib
    try:
        # For ensemble data, we need to open with the ensemble dimension
        ds = xr.open_dataset(input_file, engine='cfgrib', 
                            backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa'}})
    except Exception as e:
        print(f"Error opening GRIB file: {e}")
        return

    # Print available variables for reference
    print("Available variables:", list(ds.data_vars.keys()))
    
    # Apply subsetting
    if variables is not None:
        ds = ds[variables]
    if levels is not None and 'isobaricInhPa' in ds.dims:
        ds = ds.sel(isobaricInhPa=levels)

    
    if bbox is not None:
        min_lon, min_lat, max_lon, max_lat = bbox
        # Handle longitude wrapping if needed
        if min_lon < 0:
            min_lon += 360
        if max_lon < 0:
            max_lon += 360
            
        # Select spatial subset
        ds = ds.where(
            (ds.longitude >= min_lon) & 
            (ds.longitude <= max_lon) & 
            (ds.latitude >= min_lat) & 
            (ds.latitude <= max_lat),
            drop=True
        )

    # Add metadata
    ds.attrs['history'] = f"Processed by GFS ensemble converter on {datetime.now()}"
    ds.attrs['source'] = input_file
    
    # Save to NetCDF
    try:
        ds.to_netcdf(output_file)
        print(f"Successfully saved subset to {output_file}")
    except Exception as e:
        print(f"Error saving NetCDF file: {e}")
        

def merge_gfs_ensemble(grib_files, members , lead_times , output_nc, 
                      bbox=None, variables=None, 
                      levels=None):
    """
    Merge multiple GFS ensemble members and lead times into a single NetCDF file.
    
    Parameters:
        grib_files (list): List of GRIB file paths or glob pattern
        output_nc (str): Output NetCDF file path
        bbox (tuple): (min_lon, min_lat, max_lon, max_lat) for spatial subset
        variables (list): Variables to extract (None for all)
        levels (list): Pressure levels to extract (None for all)
        time_dim (str): Time dimension name ('step' or 'time')
    """
    try:
        
        # Open each file and combine into a single dataset
        ds_list = []
        for ii , file in enumerate( grib_files ):
            print(file)
            try:
                # Open with appropriate filters
                backend_kwargs = {
                    'filter_by_keys': {
                        'typeOfLevel': 'isobaricInhPa' if levels else None,
                        'level': levels
                    }
                }
                
                # Open single file
                ds = xr.open_dataset(file, engine='cfgrib', 
                                    backend_kwargs=backend_kwargs)
                # Apply subsetting
                if variables is not None:
                    ds = ds[variables]
                if levels is not None and 'isobaricInhPa' in ds.dims:
                    ds = ds.sel(isobaricInhPa=levels)
                # Apply spatial subset if requested
                if bbox:
                    min_lon, min_lat, max_lon, max_lat = bbox
                    # Handle longitude wrapping
                    if min_lon < 0:
                        min_lon += 360
                    if max_lon < 0:
                        max_lon += 360
            
                    ds = ds.where(
                        (ds.longitude >= min_lon) & 
                        (ds.longitude <= max_lon) & 
                        (ds.latitude >= min_lat) & 
                        (ds.latitude <= max_lat),
                        drop=True
                        )
                
                # Extract member number from filename if not in data
                if 'member' not in ds.dims :
                    ds = ds.expand_dims({'member': [int( members[ii] )]})
                if 'lead_time' not in ds.dims :
                    ds = ds.expand_dims({'lead_time':[int( lead_times[ii] )]})
                
                ds_list.append(ds)
                
            except Exception as e:
                print(f"Error processing {file}: {e}")
                continue
        
        if not ds_list:
            raise ValueError("No valid GRIB files processed")
        
        # Combine all datasets along ensemble and time dimensions
        print("Merging datasets...")
        combined = xr.combine_by_coords(
            ds_list,
            combine_attrs='drop_conflicts'
        )
        


        # Add metadata
        #combined.attrs['history'] = f"Merged by GFS processor on {datetime.now()}"
        #combined.attrs['source_files'] = grib_files
        
        # Save to NetCDF with compression
        #encoding = {var: {'zlib': True, 'complevel': 4} for var in combined.data_vars}
        print(f"Saving to {output_nc}...")
        #combined.to_netcdf(output_nc, mode = 'w' )
        print("Successfully saved merged ensemble data")
        
        return combined
    
    except Exception as e:
        print(f"Error in merge_gfs_ensemble: {e}")
        raise

In [12]:
# %%
# Example usage
ens_size = 1
date = '20231216'
init_time = '00'
data_path = '/home/jruiz/datosdemerzel/GFSDATA/gefs.' + date +'/' + init_time + '/pgrb2b/'
output_file = data_path + 'gfs_ens.nc'
ens_members = np.arange(0,ens_size+1).astype(str)    #.zfill(3)
print(ens_members)

#if __name__ == "__main__":
# Input parameters
levels = [1000.,  900.,  800.,  700.,  600.,  500.,  400.,  300.]
# Define subset parameters
bbox = (-75, -60, -40, -20 )   # Continental US approx
variables = ['t', 'q', 'u', 'v' , 'gh']  # Temperature, humidity, wind components

file_list = []
members_list = []
lead_time_list = [] 
for imem in ens_members :
    print('My member is ',imem)
    str_mem = imem.zfill(3)
    file_list += glob.glob( data_path + str_mem + '/*.pgrb2.*[!idx][!nc]' ) 
        
    members_list = []
    lead_time_list = []
    for my_file in file_list :
        members_list.append( str_mem )
        lead_time_list.append( int(my_file[-3:]) )
        print(members_list[-1],lead_time_list[-1])
    
    
# Run the merge
combined_ds = merge_gfs_ensemble(
    file_list , members_list , lead_time_list ,
    output_file,
    bbox=bbox,
    variables=variables,
    levels=levels
)


print(combined_ds)
    
    

skipping variable: paramId==260163 shortName='p260163'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
    ...<2 lines>...
    )
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='isobaricInhPa' value=Variable(dimensions=('isobaricInhPa',), data=array([1000.,  900.,  800.,  700.,  600.,  500.,  400.,  300.])) new_value=Variable(dimensions=('isobaricInhPa',), data=array([800., 700., 600., 500., 400., 300.]))
skipping variable: paramId==7001293 shortName='ICSEV'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    

skipping variable: paramId==260131 shortName='o3mr'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
    ...<2 lines>...
    )
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='isobaricInhPa' value=Variable(dimensions=('isobaricInhPa',), data=array([1000.,  900.,  800.,  700.,  600.,  500.,  400.,  300.])) new_value=Variable(dimensions=('isobaricInhPa',), data=array([400., 300.]))
skipping variable: paramId==260080 shortName='wavh5'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_

['0' '1']
My member is  0
000 0
000 24
000 6
000 15
000 18
000 3
000 21
000 33
000 12
000 36
000 30
000 9
000 27
My member is  1
001 0
001 24
001 6
001 15
001 18
001 3
001 21
001 33
001 12
001 36
001 30
001 9
001 27
001 6
001 3
001 24
001 36
001 12
001 15
001 0
001 27
001 33
001 30
001 21
001 18
001 9
/home/jruiz/datosdemerzel/GFSDATA/gefs.20231216/00/pgrb2b/000/gec00.t00z.pgrb2.0p50.f000


/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
skipping variable: paramId==260163 shortName='p260163'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_var

/home/jruiz/datosdemerzel/GFSDATA/gefs.20231216/00/pgrb2b/000/gec00.t00z.pgrb2.0p50.f024


...

/home/jruiz/datosdemerzel/GFSDATA/gefs.20231216/00/pgrb2b/001/gep01.t00z.pgrb2.0p50.f018


/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(
skipping variable: paramId==260163 shortName='p260163'
Traceback (most recent call last):
  File "/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_var

/home/jruiz/datosdemerzel/GFSDATA/gefs.20231216/00/pgrb2b/001/gep01.t00z.pgrb2.0p50.f009


/home/jruiz/anaconda3/envs/gfs-env/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


Merging datasets...
Saving to /home/jruiz/datosdemerzel/GFSDATA/gefs.20231216/00/pgrb2b/gfs_ens.nc...
Successfully saved merged ensemble data
<xarray.Dataset> Size: 12MB
Dimensions:        (lead_time: 13, member: 1, isobaricInhPa: 8, latitude: 81,
                    longitude: 71)
Coordinates:
  * lead_time      (lead_time) int64 104B 0 3 6 9 12 15 18 21 24 27 30 33 36
  * member         (member) int64 8B 1
    number         int64 8B 1
    time           datetime64[ns] 8B 2023-12-16
    step           (lead_time) timedelta64[ns] 104B 00:00:00 ... 1 days 12:00:00
  * isobaricInhPa  (isobaricInhPa) float64 64B 1e+03 900.0 800.0 ... 400.0 300.0
  * latitude       (latitude) float64 648B -20.0 -20.5 -21.0 ... -59.5 -60.0
  * longitude      (longitude) float64 568B 285.0 285.5 286.0 ... 319.5 320.0
    valid_time     (lead_time) datetime64[ns] 104B 2023-12-16 ... 2023-12-17T...
Data variables:
    t              (lead_time, member, isobaricInhPa, latitude, longitude) float32 2MB ...
    q

In [13]:
combined_ds.to_netcdf('./output_nc', mode = 'w' )